# FiberPulse AI: Day 3 reference notebook (Phase 2, real models)

**Date:** 2026-10-23. Day-end recap of what we did, with small runnable cells.
Everything is synthetic. Contents: 1) setup, 2) recap, 3) point-in-time features, 4) leakage proof,
5) model results, 6) v1 vs the stand-in v0, 7) lessons, 8) next steps.

Run this after: `uv run python -m fiberpulse.models.train --data data/synthetic`

In [ ]:
import json, shutil, sqlite3, sys, tempfile
from pathlib import Path
import pandas as pd

root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent
sys.path.insert(0, str(root / "src"))
DATA = root / "data" / "synthetic"
assert (DATA / "models" / "metrics.json").exists(), "run the train command first (see the top of this notebook)"
con = sqlite3.connect(DATA / "lakehouse.db")
print("Project root:", root)

## 2. What happened today

1. **GitHub fixed**: the Claude GitHub App now has access, so Phase 2 can be pushed on a branch (`phase-2`) and pulled by you.
2. **Phase 1 confirmed** on your machine earlier (build validated, 35 tests, day-2 notebook ran).
3. **Phase 2 built**: 72 point-in-time features, churn and acquisition models with out-of-fold scoring, a `model_scores_v1` table, a comparison report, 10 new tests (45 total).

| New file | Job |
|---|---|
| `models/features.py` | features using only data dated up to 2026-08-31 |
| `models/labels.py` | 3-month outcomes as training labels (the only bridge to `eval/`) |
| `models/train.py` | XGBoost (or scikit-learn fallback), 5-fold out-of-fold scores, optional MLflow |
| `models/report.py` | v1 vs v0 on the 50 planted accounts |

## 3. Point-in-time features
One row per account. Every value is something a model could know on 2026-08-31.

In [ ]:
from fiberpulse.models.features import build_features
X = build_features(con)
print(X.shape)
display(X[["tenure_months", "plan_speed_mbps", "contract_months_remaining", "bill_avg_aed", "late_last3",
           "tickets_90d", "open_complaints", "calls_neg", "ext_negative"]].describe().T[["mean", "min", "max"]].round(2))

## 4. Leakage proof
If we delete everything dated after the snapshot, the features must not change. This is the same check the test suite runs.

In [ ]:
tmp = Path(tempfile.mkdtemp()) / "lakehouse.db"
shutil.copy(DATA / "lakehouse.db", tmp)
c2 = sqlite3.connect(tmp)
snap = "2026-08-31"
c2.execute(f"DELETE FROM sr_complaint WHERE opened_date > '{snap}'")
c2.execute(f"UPDATE sr_complaint SET closed_date=NULL, csat=NULL, resolution_hours=NULL WHERE closed_date > '{snap}'")
c2.execute(f"DELETE FROM call_intent_signals WHERE call_date > '{snap}'")
c2.execute(f"DELETE FROM external_enrichment WHERE signal_date > '{snap}'")
c2.execute(f"DELETE FROM bills WHERE bill_month > '2026-08'")
c2.commit()
X2 = build_features(c2)
pd.testing.assert_frame_equal(X, X2)
print("Identical features after removing all post-snapshot data: no leakage from those tables.")

## 5. Model results
AUC is how well the score ranks accounts. 'Oracle' is the AUC of the hidden true probability: the ceiling.

In [ ]:
m = json.loads((DATA / "models" / "metrics.json").read_text())
print("backend:", m["backend"], "| model_version:", m["model_version"], "| features:", m["n_features"])
rows = []
for sig, d in m["signals"].items():
    x = d["metrics"]
    rows.append({"signal": sig, "AUC": x["auc"], "AUC v0": x["auc_v0_simulated"], "AUC oracle": x["auc_oracle_true_probability"],
                 "PR-AUC": x["pr_auc"], "base rate": x["base_rate"], "lift@top 6.25%": x["lift_at_top"], "Brier": x["brier"]})
display(pd.DataFrame(rows).round(3))
for sig in m["signals"]:
    print(sig, "top features:", list(m["signals"][sig]["top_features"])[:6])

## 6. Real model (v1) vs stand-in (v0)

In [ ]:
from fiberpulse.models.report import build_report
r = build_report(DATA)
display(pd.DataFrame(r["flag_overlap"]).T)
print("Planted accounts still flagged by v1:", r["planted_still_flagged"]["total"], r["planted_still_flagged"]["by_verdict"])
print("Paradox-vs-confirmed AUC  v0:", r["can_v1_tell_paradox_from_confirmed"]["auc_v0_score"],
      " v1:", r["can_v1_tell_paradox_from_confirmed"]["auc_v1_score"],
      " truth:", r["can_v1_tell_paradox_from_confirmed"]["auc_true_probability"])

## 7. Lessons from today

- **A real model can match the stand-in's skill but flag different accounts.** AUC is about the same as v0, yet only about half of the flagged accounts overlap, because the model cannot see the hidden drivers.
- **Features need an as-of date.** Ticket status, plan and contract fields in the lakehouse describe *today*; rewinding them to the snapshot is most of the work.
- **A leakage test must not share code with what it tests.** My first version read the same month constant as the feature code; injecting a deliberate leak showed it was blind, so I rewrote it with a literal.
- **Keep the grading set on the scores it was built on.** The 50 planted accounts came from v0, so agent evals stay on `model_scores`, and v1 is a second, more realistic table.
- **Cross-validation, not out-of-time.** One snapshot means no out-of-time split; your real churn work should keep using one.

## 8. Next steps
1. Run `uv run python -m fiberpulse.models.train` on your machine with XGBoost and paste the metrics (the XGBoost path is unverified on my side).
2. Optional: `--mlflow` (needs `uv sync --extra mlops`).
3. Merge `phase-2` into `main` and tag `v0.2-phase2`.
4. Phase 3: Agent 3 (Researcher) and a minimal LangGraph.